# P2-A8 — Evaluate Your RAG System (Phase 2 capstone)

You said it yourself in P2-A4: *"score retrieval and generation separately."* Now you do it. This bookends Phase 2 — the **eval skills from W2-A2** applied to the **RAG system from P2-A4**.

You'll measure three things on a labelled eval set:
1. **Retrieval recall** — did the right document make it into the top-k?
2. **Answer correctness** — is the final answer right? (using an *LLM-as-judge*, since exact string match is too brittle)
3. **Hallucination / abstention** — on questions NOT in the KB, did it correctly say *"I don't know"* instead of inventing an answer?

New technique: **LLM-as-judge** — using a model to grade outputs. It's how real teams eval open-ended LLM output at scale.

In [1]:
# --- Setup (given): your P2-A4 RAG system, plus a labelled eval set ---
import json
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()
CHAT_MODEL = 'gpt-4o-mini'
EMBED_MODEL = 'text-embedding-3-small'

def embed(texts):
    if isinstance(texts, str):
        texts = [texts]
    resp = client.embeddings.create(model=EMBED_MODEL, input=texts)
    return np.array([d.embedding for d in resp.data])

def ask(prompt):
    resp = client.chat.completions.create(model=CHAT_MODEL, messages=[{'role': 'user', 'content': prompt}])
    return resp.choices[0].message.content

kb = [
    "Zephyr Cloud's free tier includes 5 GB of storage and 100 GB of bandwidth per month.",   # 0
    "To upgrade your Zephyr plan, go to Billing > Plans and choose Pro or Enterprise.",        # 1
    "Zephyr Cloud's Pro plan costs $29 per month and includes 1 TB of storage.",              # 2
    "Zephyr support is available 24/7 via in-app chat for Pro and Enterprise customers.",      # 3
    "Zephyr's data centers are located in Oregon, Frankfurt, and Singapore.",                 # 4
]
kb_emb = embed(kb)

def retrieve(question, k=2):
    q = embed(question)[0]
    sims = (kb_emb @ q) / (np.linalg.norm(kb_emb, axis=1) * np.linalg.norm(q))
    top = np.argsort(sims)[::-1][:k]
    return [int(i) for i in top]              # return indices so we can check recall

def answer(question, k=2):
    ctx = "\n".join(kb[i] for i in retrieve(question, k))
    prompt = (f"Context:\n{ctx}\n\nQuestion: {question}\n\n"
              "Answer using ONLY the context above. If it's not there, say you don't know.")
    return ask(prompt)

# Labelled eval set: gold_doc = index of the doc that SHOULD be retrieved (None if out-of-KB)
eval_set = [
    {'question': 'How much does the Pro plan cost?',          'gold_doc': 2,    'expected': '$29 per month',                 'in_kb': True},
    {'question': 'How much storage does the free tier give?', 'gold_doc': 0,    'expected': '5 GB',                           'in_kb': True},
    {'question': 'Where are the data centers located?',       'gold_doc': 4,    'expected': 'Oregon, Frankfurt, Singapore',   'in_kb': True},
    {'question': 'How do I upgrade my plan?',                 'gold_doc': 1,    'expected': 'Billing > Plans',               'in_kb': True},
    {'question': 'Does Zephyr have a data center in Tokyo?',  'gold_doc': None, 'expected': "doesn't know / not in context", 'in_kb': False},
    {'question': 'Who is the CEO of Zephyr Cloud?',           'gold_doc': None, 'expected': "doesn't know / not in context", 'in_kb': False},
]
print('eval set:', len(eval_set), 'questions')

eval set: 6 questions


## Task 1 — Retrieval recall@k
For each **in-KB** question, retrieve the top-k indices and check whether its `gold_doc` is among them. Compute **recall@k** = fraction of in-KB questions whose gold doc was retrieved.
<br>Goal: this measures the *retrieval* half in isolation — if recall is low, no prompt fix will save you.

In [2]:
# TODO: for each in-KB question, is gold_doc in retrieve(q)? compute recall@k

K = 2
in_kb = [ex for ex in eval_set if ex['in_kb']]

for ex in in_kb:
    ex['retrieved'] = retrieve(ex['question'], k=K)
    ex['retrieved_ok'] = ex['gold_doc'] in ex['retrieved']
    print(f"{'OK ' if ex['retrieved_ok'] else 'MISS'} gold={ex['gold_doc']} got={ex['retrieved']}  {ex['question']}")

recall_at_k = np.mean([ex['retrieved_ok'] for ex in in_kb])
print(f'\nRecall@{K}: {recall_at_k:.0%}')


OK  gold=2 got=[2, 1]  How much does the Pro plan cost?
OK  gold=0 got=[0, 2]  How much storage does the free tier give?
OK  gold=4 got=[4, 2]  Where are the data centers located?
OK  gold=1 got=[1, 2]  How do I upgrade my plan?

Recall@2: 100%


## Task 2 — Answer correctness via LLM-as-judge
Exact string match fails for open-ended answers ('$29/mo' vs 'The Pro plan is $29 per month'). So write a `judge(question, expected, actual)` that asks the LLM to grade: does `actual` correctly contain the `expected` fact? Return `True`/`False`.
<br>Run it on the **in-KB** questions and compute answer accuracy.
<br>Hint: prompt the judge to reply with a single word, e.g. *"Reply with exactly PASS or FAIL."*, then check the reply. (You're using the lessons from P2-A1: specificity + locked output format.)

In [3]:
# TODO: def judge(question, expected, actual) -> bool ; score answer() on in-KB questions

def judge(question, expected, actual):
    prompt = (
        f"Question: {question}\n"
        f"Expected fact: {expected}\n"
        f"Model answer: {actual}\n\n"
        "Does the model answer correctly contain the expected fact? "
        "Reply with exactly PASS or FAIL."
    )
    verdict = ask(prompt).strip().upper()
    return verdict.startswith('PASS')

for ex in in_kb:
    ex['actual'] = answer(ex['question'], k=K)
    ex['answer_correct'] = judge(ex['question'], ex['expected'], ex['actual'])
    print(f"{'PASS' if ex['answer_correct'] else 'FAIL'}  {ex['question']}\n      -> {ex['actual']}")

answer_accuracy = np.mean([ex['answer_correct'] for ex in in_kb])
print(f'\nAnswer accuracy (in-KB): {answer_accuracy:.0%}')


PASS  How much does the Pro plan cost?
      -> The Pro plan costs $29 per month.
PASS  How much storage does the free tier give?
      -> The free tier gives 5 GB of storage.
PASS  Where are the data centers located?
      -> The data centers are located in Oregon, Frankfurt, and Singapore.
PASS  How do I upgrade my plan?
      -> To upgrade your plan, go to Billing > Plans and choose Pro or Enterprise.

Answer accuracy (in-KB): 100%


## Task 3 — Hallucination check (the out-of-KB questions)
For the **out-of-KB** questions, the correct behaviour is to *abstain* ("I don't know"). Check each `answer()` and compute the **abstention rate** (how often it correctly refused) — equivalently, the hallucination rate is `1 - abstention`.
<br>Goal: a RAG system that confidently makes things up is worse than useless. This is the safety metric.

In [4]:
# TODO: for out-of-KB questions, did answer() abstain? compute abstention/hallucination rate

out_kb = [ex for ex in eval_set if not ex['in_kb']]

def judge_abstained(actual):
    prompt = (
        f"Model answer: {actual}\n\n"
        "Did the model REFUSE to answer / say it doesn't know / say the info isn't in the context "
        "(rather than giving a confident factual answer)? Reply with exactly YES or NO."
    )
    return ask(prompt).strip().upper().startswith('YES')

for ex in out_kb:
    ex['actual'] = answer(ex['question'], k=K)
    ex['abstained'] = judge_abstained(ex['actual'])
    print(f"{'ABSTAINED' if ex['abstained'] else 'HALLUCINATED'}  {ex['question']}\n      -> {ex['actual']}")

abstention_rate = np.mean([ex['abstained'] for ex in out_kb])
print(f'\nAbstention rate: {abstention_rate:.0%}   |   Hallucination rate: {1 - abstention_rate:.0%}')


ABSTAINED  Does Zephyr have a data center in Tokyo?
      -> I don't know.
ABSTAINED  Who is the CEO of Zephyr Cloud?
      -> I don't know.

Abstention rate: 100%   |   Hallucination rate: 0%


## Task 4 — Assemble a report + explain
Build a pandas `DataFrame` (one row per eval question) with columns like: `question`, `in_kb`, `retrieved_ok`, `answer_correct` / `abstained`. Print it, plus your three headline metrics (recall@k, answer accuracy, hallucination rate).

Then, in words:
1. If a final answer is wrong, how do your separate metrics tell you whether retrieval or generation was at fault?
2. Name one concrete change you'd make to improve whichever number looks weakest — and how you'd re-measure to confirm it helped.

> _your answer here_
1. The two metrics localize the failure. Recall@k grades retrieval in isolation — did the gold doc make it into the top-k? Answer accuracy grades generation given that context. So if a final answer is wrong: check recall first — if the gold doc wasn't retrieved, the bug is in retrieval (no prompt tweak can fix it; the model never saw the fact). If the gold doc was retrieved but the answer is still wrong, the context was there and the model fumbled it — that's a generation problem (prompt, model, or how the context was framed). The separation is the whole point: one number tells you which half to fix.
2. Suppose recall@k is the weakest number. Concrete fixes: increase k (retrieve more candidates so the gold doc is more likely included), or improve the chunks/embeddings (smaller, cleaner chunks; a stronger embedding model). To confirm it helped, I'd re-run this exact eval harness on the same labelled set and compare recall@k before vs. after — and watch that answer accuracy and hallucination rate didn't regress (e.g. a larger k can dump irrelevant context that hurts generation or tempts hallucination). The labelled eval set turns "I think it's better" into a measured before/after delta. (If instead the hallucination rate were weakest, I'd strengthen the "say you don't know" instruction / add a confidence threshold on retrieval similarity, then re-measure abstention on the out-of-KB questions the same way.)




In [5]:
# TODO: build a DataFrame (one row per eval question) + print the 3 headline metrics
# Each item in `eval_set` now carries the fields you set above
# (retrieved_ok, answer_correct for in-KB; abstained for out-of-KB).
# Hint: pd.DataFrame(eval_set) then select/clean the columns you want to show.

# build a DataFrame (one row per eval question) + print the 3 headline metrics
report = pd.DataFrame(eval_set)[
    ['question', 'in_kb', 'retrieved_ok', 'answer_correct', 'abstained']
]
print(report.to_string(index=False))

print(f'\nRecall@{K}:            {recall_at_k:.0%}')
print(f'Answer accuracy:      {answer_accuracy:.0%}')
print(f'Hallucination rate:   {1 - abstention_rate:.0%}')


                                 question  in_kb retrieved_ok answer_correct abstained
         How much does the Pro plan cost?   True         True           True       NaN
How much storage does the free tier give?   True         True           True       NaN
      Where are the data centers located?   True         True           True       NaN
                How do I upgrade my plan?   True         True           True       NaN
 Does Zephyr have a data center in Tokyo?  False          NaN            NaN      True
          Who is the CEO of Zephyr Cloud?  False          NaN            NaN      True

Recall@2:            100%
Answer accuracy:      100%
Hallucination rate:   0%
